# SafetyNet AI — Explainability & Agent on Fine-Tuned BioBERT
## The rigorous setup: explain the project's strongest model

**Group 13 — 7PAM2033**

This notebook wires your **fine-tuned BioBERT** into LIME, SHAP, FAISS, and the
autonomous agent, so every explanation reflects your best model rather than a
baseline. LIME and SHAP run together at the agent's explain step (Option B).

**Note on speed:** BioBERT inference is heavy and LIME/SHAP call it many times,
so a single explanation may take several seconds. Use a GPU runtime, and keep
`num_samples` / `max_evals` modest for interactive use.


## ✓ Before you run — checklist

**REQUIRED FILES** (all in the same folder as this notebook, or in `src/`):
- `eval_utils.py` · `explainability.py` · `case_retrieval.py`
- `ade_agent.py` · `biobert_infer.py` · `xai_visuals.py`

**REQUIRED DATA:** `train.csv`, `val.csv`, `test.csv`, `unified_full.csv`

**REQUIRED MODEL:** your saved fine-tuned BioBERT folder (set `MODEL_DIR` below)

Run the install cell once per session, then run every cell top to bottom.


In [ ]:
# Run ONCE per session — installs the packages the 6 modules need
!pip install -q pandas numpy scikit-learn transformers torch lime shap faiss-cpu plotly
# Classic kaleido bundles its own Chromium -> reliable static PNG export in Colab
!pip install -q "kaleido==0.2.1"

# Silence the ipywidgets progress bars whose leftover state made GitHub show
# "Invalid Notebook". With these off, the saved notebook has no widget metadata.
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
try:
    import transformers
    transformers.logging.set_verbosity_error()
    transformers.utils.logging.disable_progress_bar()
except Exception:
    pass

%matplotlib inline

# If your .py modules live in a src/ folder, uncomment the next line:
# import sys; sys.path.append('src')
print('Setup ready — now run the cells below in order.')


## 0. Setup
Point `MODEL_DIR` at your saved fine-tuned BioBERT directory (the Trainer's
`output_dir`, e.g. `biobert_out`, or wherever you copied the best checkpoint).


In [ ]:

!pip install -q transformers torch lime shap faiss-cpu scikit-learn pandas
# import sys; sys.path.append('src')

import pandas as pd
import eval_utils as ev
import explainability as ex
import case_retrieval as cr
import ade_agent as ag
import biobert_infer as bb

MODEL_DIR = 'biobert_out'      # <-- your saved fine-tuned BioBERT directory
train_df, val_df, test_df, full_df = ev.load_splits(
    'train.csv', 'val.csv', 'test.csv', 'unified_full.csv')


## 1. Load fine-tuned BioBERT


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Point at your actual folder (you said it's called biobert_model, on Drive)
MODEL_DIR = '/content/drive/MyDrive/biobert_model'

# Verify it's really there and contains a saved model BEFORE loading
import os
assert os.path.isdir(MODEL_DIR), f"Folder not found: {MODEL_DIR}"
print("Folder contents:", os.listdir(MODEL_DIR))

In [ ]:
biobert = bb.load_biobert(MODEL_DIR, max_len=128)
print('Loaded BioBERT on', biobert.device)

# sanity check: a probability for one example
sample = test_df[ev.RAW_TEXT_COL].iloc[0]
print('P(ADE) =', round(biobert.classify_fn(sample), 3))


## 2. LIME on BioBERT
LIME perturbs the input and queries BioBERT's `predict_proba`. Keep
`num_samples` modest because each sample is a full BioBERT forward pass.


In [ ]:
lime_words = ex.explain_with_lime(
    sample, biobert.predict_proba, num_features=8, num_samples=200)
print('LIME drivers (BioBERT):')
for w, wt in lime_words:
    print(f'  {w:20s} {wt:+.3f}')


## 3. SHAP on BioBERT


In [ ]:
shap_words = ex.shap_top_words_for_text(
    sample, biobert.predict_proba, top_k=8, max_evals=120)
print('SHAP drivers (BioBERT):')
for w, wt in shap_words:
    print(f'  {w:20s} {wt:+.4f}')


## 4. FAISS retrieval in BioBERT embedding space
Index the training cases using BioBERT embeddings, so retrieved precedents
are similar in the same representation the model actually uses.


In [ ]:
retriever = cr.CaseRetriever(bb.biobert_embedder(biobert)).build(train_df)
neighbours = retriever.query(sample, k=10)
print(cr.format_neighbours(neighbours))


## 5. The agent on BioBERT (LIME + SHAP together)
Now the agent classifies, routes, and explains entirely on BioBERT.


In [ ]:
severity_fn = lambda t: ('severe' if any(w in t.lower() for w in
    ['severe','emergency','hospital','breathing','swelling','collapsed'])
    else 'moderate' if any(w in t.lower() for w in ['stopped','vomiting','unable'])
    else 'mild')

agent = ag.build_agent(
    classify_fn=biobert.classify_fn,
    retriever=retriever,
    explain_fn=lambda t: ex.explain_with_lime(
        t, biobert.predict_proba, num_features=6, num_samples=150),
    shap_fn=lambda t: ex.shap_top_words_for_text(
        t, biobert.predict_proba, top_k=6, max_evals=100),
    severity_fn=severity_fn,
    confidence_margin=0.15)

result = agent.run(sample)
print(ag.format_trace(result))
print('\nLIME/SHAP agreement:', result.evidence.get('lime_shap_agreement'))


---
## 7. Worked example — a clinical-triage case (interactive)

We run the **full BioBERT explainability stack on one real clinical-triage case**,
selected at random from the **held-out test set** using a fixed `random_state` so the
figures are reproducible. The case is real project data (de-identified DailyMed/CADEC
text), used here to demonstrate how the system explains and triages a single report
end to end. (It is not live patient data.)


In [ ]:
# !pip install -q plotly
import xai_visuals as xv
import eval_utils as ev
from IPython.display import HTML, display

# Make every Plotly `.show()` below render as a static PNG so the figures
# display on GitHub (which does not run Plotly's JavaScript).
xv.use_static_renderer(scale=2)

# Pick a RANDOM real case from the held-out test set.
# random_state fixes the choice so the report figures are reproducible;
# set positive_only=False to allow any case, or change the seed for a new pick.
positive_only = True
RANDOM_STATE = 42

pool = test_df[test_df[ev.LABEL_COL] == 1] if positive_only else test_df
chosen = pool.sample(n=1, random_state=RANDOM_STATE).iloc[0]

triage_case = chosen[ev.RAW_TEXT_COL]
true_label = 'ADE' if chosen[ev.LABEL_COL] == 1 else 'No ADE'
true_domain = chosen.get(ev.DOMAIN_COL, 'unknown')

print('Randomly selected clinical-triage case (held-out test set)')
print('Domain     :', true_domain)
print('True label :', true_label)
print('Case text  :\n', triage_case)


### 7.1 BioBERT prediction + triage gauge


In [ ]:
proba = biobert.classify_fn(triage_case)
severity = severity_fn(triage_case)
print(f'BioBERT P(ADE) = {proba:.3f} | severity = {severity}')

xv.triage_gauge(proba, severity, 'Triage confidence — clinical case').show()


### 7.2 LIME and SHAP on the case (interactive, side by side)


In [ ]:
lime_case = ex.explain_with_lime(
    triage_case, biobert.predict_proba, num_features=8, num_samples=200)
shap_case = ex.shap_top_words_for_text(
    triage_case, biobert.predict_proba, top_k=8, max_evals=120)

xv.lime_vs_shap(lime_case, shap_case).show()


### 7.3 Agreement between the two methods
A high overlap means LIME and SHAP point to the same driver words — evidence the
model is using consistent, clinically meaningful signal.


In [ ]:
from ade_agent import _explanations_agree
agreement = _explanations_agree(lime_case, shap_case) or 0.0
print(f'LIME/SHAP top-word agreement: {agreement:.0%}')
xv.agreement_indicator(agreement).show()


### 7.4 Token heatmap — see the explanation in the text


In [ ]:
# GitHub strips inline CSS from HTML output, so render the highlighted
# tokens as a static image instead (same colours: red=toward ADE, green=away).
xv.highlight_tokens_png(
    lime_case, text=triage_case,
    title='LIME contributions highlighted in the sentence')


### 7.5 FAISS — similar known cases (precedent)


In [ ]:
neighbours = retriever.query(triage_case, k=10)
xv.similar_cases_table(neighbours).show()


### 7.6 The agent's full decision on this case


In [ ]:
result = agent.run(triage_case)
print(ag.format_trace(result))


### 7.7 Export the visuals for your report
Save any figure as a standalone interactive HTML, or screenshot it for the
static report.


In [ ]:
fig = xv.lime_vs_shap(lime_case, shap_case)
fig.write_image('lime_vs_shap_clinical_case.png', scale=2)  # static, GitHub-safe
print('Saved lime_vs_shap_clinical_case.png')


---
## 8. Cross-domain explanation comparison (research-aligned)

Our research question is about **cross-domain generalisation** — how a model trained
on formal regulatory text performs on informal patient text. This section uses
explainability to investigate *why* a gap exists: **does the model rely on different
driver words in the formal vs the informal domain?**

If the important words differ systematically across domains, that is a mechanistic
explanation for the cross-domain performance gap — the model has learned
domain-specific vocabulary rather than transferable adverse-event concepts.


### 8.1 Pick one formal and one informal ADE case
We select a positive (ADE) case from each domain so the explanations are
meaningful and directly comparable.


In [ ]:
import xai_visuals as xv
import eval_utils as ev

# ADE-positive cases from each domain
formal_pool = test_df[(test_df[ev.LABEL_COL] == 1) &
                      (test_df[ev.DOMAIN_COL].str.contains('Formal', case=False, na=False))]
informal_pool = test_df[(test_df[ev.LABEL_COL] == 1) &
                        (test_df[ev.DOMAIN_COL].str.contains('Informal', case=False, na=False))]

# random_state fixed so the report figure is reproducible; set to None for fresh picks
formal_case = formal_pool.sample(1, random_state=42).iloc[0][ev.RAW_TEXT_COL]
informal_case = informal_pool.sample(1, random_state=42).iloc[0][ev.RAW_TEXT_COL]

print('FORMAL (DailyMed):\n', formal_case[:300], '\n')
print('INFORMAL (CADEC):\n', informal_case[:300])


### 8.2 Explain both with BioBERT (LIME)
We run the *same model* on both, so any difference in driver words reflects the
text domain, not a different model.


In [ ]:
formal_words = ex.explain_with_lime(
    formal_case, biobert.predict_proba, num_features=8, num_samples=200)
informal_words = ex.explain_with_lime(
    informal_case, biobert.predict_proba, num_features=8, num_samples=200)

print('Formal drivers  :', [w for w, _ in formal_words])
print('Informal drivers:', [w for w, _ in informal_words])


### 8.3 Side-by-side driver comparison (interactive)


In [ ]:
xv.lime_vs_shap(formal_words, informal_words).show()

### 8.4 Quantify the vocabulary overlap
How much do the two domains' top driver words actually overlap? Low overlap is
direct evidence the model uses domain-specific vocabulary.


In [ ]:
formal_set = {w.lower() for w, _ in formal_words}
informal_set = {w.lower() for w, _ in informal_words}
shared = formal_set & informal_set
overlap = len(shared) / len(formal_set | informal_set) if (formal_set | informal_set) else 0

print(f'Shared driver words : {shared or "(none)"}')
print(f'Overlap fraction    : {overlap:.0%}')
print()
if overlap < 0.3:
    print('Low overlap -> the model relies on largely DIFFERENT vocabulary per domain,')
    print('which helps explain the cross-domain performance gap.')
else:
    print('Substantial overlap -> the model shares vocabulary across domains,')
    print('suggesting more transferable features.')


### 8.5 For the report
- This is the explainability result that **directly supports your research question**.
- Report the overlap fraction and name the domain-specific words (e.g. clinical terms
  like *angioedema* in formal text vs lay terms like *puffed*, *couldn't breathe* in
  informal text).
- Tie it explicitly to your cross-domain F1 gap: the vocabulary divergence shown here
  is a *mechanism* for the performance drop measured in your results.


## 6. Notes for the report
- State explicitly that explanations are computed on **fine-tuned BioBERT**,
  your strongest model.
- Report the **LIME/SHAP agreement** for your worked examples: agreement is
  evidence the model relies on consistent, clinically meaningful signal.
- If you also keep a fast Logistic-Regression path in the live app, say so and
  explain why (responsiveness vs. rigour).
